# oss-backtest-kit quick start

Everything below uses **synthetic data** (a seeded random walk). Nothing is downloaded and nothing here is a trading result or investment advice.

The question: you tried several variants of a strategy and kept the best one. How much of its Sharpe ratio is just luck?

In [ ]:
# Install the released package from PyPI (pinned to the tested version).
%pip install -q "oss-backtest-kit==0.2.0"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from oss_backtest_kit import stats
from oss_backtest_kit.engine import EngineConfig, MarketData, run_backtest

## 1. A synthetic market (two random-walk symbols, fixed seed)

In [ ]:
DAY = 86_400_000
n_bars, symbols = 1000, ["AAA", "BBB"]
rng = np.random.default_rng(0)
rets = rng.normal(0.0003, 0.02, (n_bars, len(symbols)))
close = 100.0 * np.cumprod(1.0 + rets, axis=0)
open_ = np.vstack([np.full((1, len(symbols)), 100.0), close[:-1]])
high, low = np.maximum(open_, close) * 1.005, np.minimum(open_, close) * 0.995
times = 1_600_000_000_000 - (1_600_000_000_000 % DAY) + DAY * np.arange(n_bars)
funding_px = 0.0001 * open_  # a small constant funding rate
data = MarketData(symbols, times, open_, high, low, close, funding_px)

## 2. A strategy and one backtest

The strategy is called at each bar's close and sees only bars up to then; orders fill at the next open with slippage and fees. It returns target weights.

In [ ]:
def ma_crossover(fast, slow):
    def weights(t, hist):
        if t + 1 < slow:
            return {}
        up = [s for i, s in enumerate(hist.symbols)
              if hist.close[t - fast + 1:t + 1, i].mean() > hist.close[t - slow + 1:t + 1, i].mean()]
        return {s: 1.0 / len(hist.symbols) for s in up}
    return weights

cfg = EngineConfig(symbols=symbols, max_leverage=1.0, rebalance_threshold=0.02)
result = run_backtest(data, ma_crossover(10, 40), cfg, cost_level="stressed")
{k: round(v, 4) for k, v in result.summary().items() if isinstance(v, float)}

## 3. Try 9 variants, keep the best, then deflate

PSR is the probability that the true Sharpe is above zero. DSR also charges you for how many variants you tried, so it is lower.

In [ ]:
grid = [(f, s) for f in (5, 10, 20) for s in (30, 40, 60)]
runs = [run_backtest(data, ma_crossover(f, s), cfg) for f, s in grid]
sharpes = [stats.sharpe(r.returns) for r in runs]
best = int(np.argmax(sharpes))
returns = runs[best].returns
dsr_value, benchmark, _ = stats.dsr(returns, len(grid), sharpes)
print("best variant (fast, slow):", grid[best])
print("PSR vs 0 : %.3f" % stats.psr(returns))
print("DSR (%d trials): %.3f" % (len(grid), dsr_value))

## 4. Plot the equity curve (synthetic data, not a trading result)

In [ ]:
eq = runs[best].equity
plt.figure(figsize=(8, 4))
plt.plot(eq / eq[0])
plt.axhline(1.0, color="grey", ls="--", lw=0.8)
plt.title("Best of %d variants: PSR %.2f, DSR %.2f (synthetic data, not a trading result)"
          % (len(grid), stats.psr(returns), dsr_value), fontsize=9)
plt.xlabel("bar")
plt.ylabel("equity (start = 1.0)")
plt.show()